In [ ]:
from google.colab import drive, files, userdata
import json, os, pathlib, shutil, subprocess, sys, zipfile

drive.mount('/content/drive')
try:
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'


In [ ]:
EXPECTED_ZIP = 'counterfactual_monitorability_final_diagnostics_v2_bundle_PATCHED.zip'
uploaded = files.upload()
if EXPECTED_ZIP not in uploaded:
    raise FileNotFoundError(f'Please upload exactly {EXPECTED_ZIP}')

extract_root = pathlib.Path('/content/final_diagnostics_bundle')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(EXPECTED_ZIP) as archive:
    root = extract_root.resolve()
    for member in archive.infolist():
        target = (extract_root / member.filename).resolve()
        if root not in target.parents and target != root:
            raise RuntimeError(f'Unsafe archive member: {member.filename}')
    archive.extractall(extract_root)

                                                                                     
marker_paths = sorted(extract_root.rglob('PACKAGE_PLAN_CROSSCHECK.json'))
if len(marker_paths) != 1:
    print('Extracted top-level entries:', [p.name for p in extract_root.iterdir()])
    print('Marker candidates:', [str(p) for p in marker_paths[:20]])
    raise RuntimeError(f'Could not uniquely locate bundle root; found {len(marker_paths)} marker files')

bundle_root = marker_paths[0].parent
expected_name = 'counterfactual_monitorability_final_diagnostics_v2'
if bundle_root.name != expected_name:
    print(f'Using detected bundle root {bundle_root}; expected folder name was {expected_name}')
print('Bundle root:', bundle_root)


In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    '-r', str(bundle_root / 'requirements.txt')
])
print('Pinned runtime installed.')


In [ ]:
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA'
gpu_bytes = torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0
print('GPU:', gpu_name, f'({gpu_bytes / 2**30:.1f} GiB)')
if 'A100' not in gpu_name.upper():
    raise RuntimeError('This final package is frozen for an NVIDIA A100 runtime')

config = json.loads((bundle_root / 'config' / 'standardized_diagnostics_config.json').read_text())
crosscheck = json.loads((bundle_root / 'PACKAGE_PLAN_CROSSCHECK.json').read_text())
assert crosscheck['all_models_total'] == 3588
assert crosscheck['manual_review_rows'] == 3132
output_root = pathlib.Path('/content/drive/MyDrive') / config['drive_relative_root']
output_root.mkdir(parents=True, exist_ok=True)
print('Drive output:', output_root)
print('Progress heartbeat:', output_root / 'logs' / 'progress.json')
print('Run log:', output_root / 'logs' / 'run.log')
print('Latest error:', output_root / 'logs' / 'latest_error.txt')


In [ ]:
command = [
    sys.executable,
    str(bundle_root / 'src' / 'run_all_standardized_diagnostics.py'),
    '--bundle-root', str(bundle_root),
    '--output-root', str(output_root),
    '--models', 'gemma_e2b', 'gpt_oss_20b', 'qwen35_9b',
]
print('Starting final collection. Drive heartbeat updates every 30 seconds.')
subprocess.check_call(command)


In [ ]:
marker_path = output_root / 'FINAL_COLLECTION_COMPLETE.json'
manifest_path = output_root / 'ANALYSIS_MANIFEST.json'
if not marker_path.exists() or not manifest_path.exists():
    raise RuntimeError('Final completion markers are missing')
marker = json.loads(marker_path.read_text())
manifest = json.loads(manifest_path.read_text())
assert marker['collection_complete'] is True
assert manifest['diagnostic_units']['total'] == 3588
assert manifest['manual_review']['rows'] == 3132
print(json.dumps(marker, indent=2))
print('\nFinal result folder:', output_root)
print('Model generation is complete. Manual coding and analysis remain.')
